<a href="https://colab.research.google.com/github/brevancampus/planogram_intern/blob/main/training25/3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Install YOLOv8
!pip install ultralytics

# 2. Mounting Google Drive
from google.colab import drive
import os
drive.mount('/content/drive')

# 3. Set Kaggle Token (Pakai token yang kamu dapat tadi)
os.environ['KAGGLE_API_TOKEN'] = "KGAT_1fce91d9350daa0663dab79ec06a991b"

# 4. Download SKU110K & Ekstrak dataset 6 produk
print("Sedang menarik data dari Kaggle dan Drive...")
!kaggle datasets download -d thedatasith/sku110k-annotations
!unzip -q sku110k-annotations.zip -d /content/sku110k
!unzip -q /content/drive/MyDrive/dataset_6_produk.zip -d /content/dataset_6_produk

print("[OK] Semua bahan baku sudah siap di folder Colab!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.1 MB/s eta 0:00:00
Mounted at /content/drive
Sedang menarik data dari Kaggle dan Drive...
Dataset URL: https://www.kaggle.com/datasets/thedatasith/sku110k-annotations
License(s): Attribution-NonCommercial-ShareAlike 3.0 IGO (CC BY-NC-SA 3.0 IGO)
100% 13.2G/13.2G [01:52<00:00, 126MB/s]

[OK] Semua bahan baku sudah siap di folder Colab!


In [3]:
# Download Dataset 6 Produk Resmi dari link Kaggle kamu
print("Mendownload dataset 6 produk dari Kaggle...")
!kaggle datasets download -d hafizyusufheraldi/retail-product-dataset
!unzip -q retail-product-dataset.zip -d /content/retail_dataset

print("[OK] Dataset 6 Produk berhasil didownload dan diekstrak!")

Mendownload dataset 6 produk dari Kaggle...
Dataset URL: https://www.kaggle.com/datasets/hafizyusufheraldi/retail-product-dataset
License(s): CC0-1.0
100% 13.9M/13.9M [00:00<00:00, 34.0MB/s]

[OK] Dataset 6 Produk berhasil didownload dan diekstrak!


In [7]:
# ubah format xml ke txt (yolo v8)

import os, shutil, random
import xml.etree.ElementTree as ET
from pathlib import Path

base_dir = '/content/merged_planogram'
for s in ['train', 'val']:
    for d in ['images', 'labels']:
        os.makedirs(f'{base_dir}/{d}/{s}', exist_ok=True)

# Map 6 kelas target kamu
classes = ['aqua', 'chitato', 'indomie', 'pepsodent', 'shampoo', 'tissue']
class_map = {c: i for i, c in enumerate(classes)}

print("Mencari dan memproses Dataset 6 Produk (Konversi XML ke YOLO)...")
retail_path = Path('/content/retail_dataset')
all_xmls = list(retail_path.rglob('*.xml'))

paired_data = []
for xml_file in all_xmls:
    img_name = xml_file.stem
    img_file = None

    # Assuming XMLs are in /content/retail_dataset/dataset/annotations/{mode}/{name}.xml
    # Images should be in /content/retail_dataset/dataset/images/{mode}/{name}.jpg

    # Get the parent directory of the XML file (e.g., 'test', 'train', 'val')
    # This assumes the immediate parent of the XML file is the mode directory (e.g., 'test')
    mode_dir = xml_file.parent.name

    # Construct the potential image directory path
    # This assumes retail_path is '/content/retail_dataset' and the structure is /retail_path/dataset/images/{mode_dir}
    potential_img_base_dir = retail_path / 'dataset' / 'images' / mode_dir

    # Cari pasangan gambar
    for ext in ['.jpg', '.jpeg', '.png', '.JPG']:
        potensial_img = potential_img_base_dir / (img_name + ext)
        if potensial_img.exists():
            img_file = potensial_img
            break

    if img_file:
        paired_data.append((img_file, xml_file))

random.shuffle(paired_data)
split_idx = int(len(paired_data) * 0.8)

for i, (img_path, xml_path) in enumerate(paired_data):
    mode = 'train' if i < split_idx else 'val'

    # 1. Copy Gambar
    new_img_name = f"c_{img_path.name}"
    shutil.copy(img_path, f'{base_dir}/images/{mode}/{new_img_name}')

    # 2. Convert XML ke YOLO
    tree = ET.parse(xml_path)
    root = tree.getroot()
    try:
        w = float(root.find('size/width').text)
        h = float(root.find('size/height').text)
    except:
        continue

    out_txt_path = f'{base_dir}/labels/{mode}/c_{img_path.stem}.txt'
    with open(out_txt_path, 'w') as out_file:
        for obj in root.iter('object'):
            cls_name = obj.find('name').text.lower().strip()
            if 'aqua' in cls_name: cls_name = 'aqua'
            elif 'chitato' in cls_name: cls_name = 'chitato'
            elif 'indomie' in cls_name: cls_name = 'indomie'
            elif 'pepsodent' in cls_name: cls_name = 'pepsodent'
            elif 'shampoo' in cls_name or 'sampo' in cls_name: cls_name = 'shampoo'
            elif 'tissue' in cls_name or 'tisu' in cls_name: cls_name = 'tissue'

            if cls_name not in class_map: continue

            cls_id = class_map[cls_name]
            xmlbox = obj.find('bndbox')
            b = (float(xmlbox.find('xmin').text), float(xmlbox.find('xmax').text),
                 float(xmlbox.find('ymin').text), float(xmlbox.find('ymax').text))

            x_center = ((b[0] + b[1]) / 2.0) / w
            y_center = ((b[2] + b[3]) / 2.0) / h
            width = (b[1] - b[0]) / w
            height = (b[3] - b[2]) / h
            out_file.write(f"{cls_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")

print("Memproses SKU110K (Class 6 - General Products)...")
for mode in ['train', 'val']:
    sku_img_dir = Path(f'/content/sku110k/images/{mode}')
    sku_lbl_dir = Path(f'/content/sku110k/labels/{mode}')
    if sku_img_dir.exists():
        for img_path in sku_img_dir.glob('*.*'):
            lbl_path = sku_lbl_dir / (img_path.stem + '.txt')
            if lbl_path.exists():
                shutil.copy(img_path, f'{base_dir}/images/{mode}/s_{img_path.name}')

                with open(lbl_path, 'r') as f:
                    lines = f.readlines()
                with open(f'{base_dir}/labels/{mode}/s_{lbl_path.name}', 'w') as f:
                    for line in lines:
                        parts = line.strip().split()
                        if parts:
                            parts[0] = '6'
                            f.write(' '.join(parts) + '\n')

print("[OK] Dataset Kaggle berhasil diconvert, digabung, dan disiapkan!")

Mencari dan memproses Dataset 6 Produk (Konversi XML ke YOLO)...
Memproses SKU110K (Class 6 - General Products)...
[OK] Dataset Kaggle berhasil diconvert, digabung, dan disiapkan!


In [9]:
import yaml
from ultralytics import YOLO

# ==========================================
# 1. PERSIAPAN DATA.YAML
# ==========================================
config = {
    'path': '/content/merged_planogram',
    'train': 'images/train',
    'val': 'images/val',
    'nc': 7,
    'names': ['aqua', 'chitato', 'indomie', 'pepsodent', 'shampoo', 'tissue', 'general_product']
}

with open('/content/merged_planogram/data.yaml', 'w') as f:
    yaml.dump(config, f)

print("[OK] File data.yaml berhasil disiapkan!")
print("[START] Memanaskan mesin GPU T4 untuk Training V3 (Augmentasi Ekstrem)...")

# ==========================================
# 2. UPGRADE MODEL (NANO -> SMALL)
# ==========================================
# Kita naik kelas ke 'yolov8s.pt' (Small). Kapasitas "otak"-nya
# lebih besar untuk menangkap pola bungkus produk yang rumit.
model = YOLO('yolov8s.pt')

# ==========================================
# 3. PROSES TRAINING & AUGMENTASI
# ==========================================
results = model.train(
    data='/content/merged_planogram/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,          # Diturunkan dari 32 ke 16 agar GPU tidak error "Out of Memory" untuk model 'Small'
    device=0,
    amp=True,
    patience=15,       # Rem otomatis diperpanjang jadi 15 epoch agar AI punya waktu adaptasi dengan gambar yang "disiksa"

    # --- MESIN AUGMENTASI (SIMULASI DUNIA NYATA) ---
    hsv_h=0.015,       # Variasi warna (tetap kecil agar warna asli merek tidak hilang)
    hsv_s=0.7,         # Variasi ketajaman warna
    hsv_v=0.6,         # Variasi pencahayaan (penting! Rak minimarket bawah biasanya gelap)
    degrees=15.0,      # Rotasi foto (simulasi tangan salesman yang miring saat memotret)
    perspective=0.001, # Simulasi sudut pandang 3D (karena foto kadang diambil dari bawah/atas rak)
    scale=0.5,         # Simulasi jarak foto (AI belajar mengenali produk dari jauh dan dekat)
    erasing=0.4,       # Tutupi sebagian kecil gambar (simulasi produk terhalang label harga / price tag)
    mosaic=1.0,        # Wajib untuk planogram: menggabungkan 4 foto jadi 1 agar AI terbiasa melihat barang berjejer rapat
    mixup=0.1,         # Menimpa gambar secara transparan (memperkuat insting AI)

    # --- LOKASI PENYIMPANAN ---
    project='/content/drive/MyDrive/Planogram_Models',
    name='v3_augmentasi_ekstrem' # Nama folder baru supaya file sebelumnya tidak tertimpa
)

print("\n[SELESAI] Training V3 selesai! File best.pt sudah aman di Google Drive kamu.")

[OK] File data.yaml berhasil disiapkan!
[START] Memanaskan mesin GPU T4 untuk Training V3 (Augmentasi Ekstrem)...
Ultralytics 8.4.27 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/merged_planogram/data.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.6, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, n